# 08 · Restoring for the machine: DnCNN fine-tuned with a YOLO feature loss

**Part B of *Restoring for the Machine*.** The design and hypotheses H7–H10 were frozen in [`PLAN.md`](../PLAN.md) §2
after Part A and before this notebook ran.

**The idea.** Notebook 07 found that DnCNN removes 82–93% of the pixel error from noisy images but only ~60% of the
damage that reaches YOLOv8n's detection layers. Here we fine-tune DnCNN to remove that too:

  L = MSE / MSE₀ + λ · D / D₀,   D = cosine distance between YOLOv8n's layer-21 (P5) features of the restored and the
  clean image (Part A's pre-registered pick), YOLOv8n frozen.

| Section | What |
|---|---|
| 1–3 | splits, detectors, and checks on the differentiable YOLO forward |
| 4–5 | loss scales, then the λ sweep (λ = 0 is the control) |
| 6 | choosing λ\* on the validation split |
| 7 | the neck-mean loss at λ\* (comparison) |
| 8 | test: PSNR and mAP for YOLOv8n, YOLOv8s and Faster R-CNN at σ = 0, 15, 25, 50 |
| 9 | results, bootstrap CIs, scorecard |
| 10 | clean-up: deletes COCO so the output holds only the results |

**Requirements:** Kaggle, **GPU T4**, **Internet ON**. About 4–5 h as a commit (Save Version → Save & Run All).
Set `QUICK = True` first for a ~15-minute check.
**Inputs:** the **dcnn-last** dataset (`shourjabanerjee/dcnn-last`). The code is cloned from GitHub.
**Outputs** (`/kaggle/working/outputs/08`): checkpoints, training logs, `runs.csv`, `per_image.csv`,
`comparisons.csv`, `scorecard_08.json`, figures.

In [ ]:
!pip -q install ultralytics pycocotools

In [ ]:
import os, sys, glob, json, time, subprocess

REPO_URL = "https://github.com/onigiri4711/restoring-for-the-machine"
attached = sorted(glob.glob("/kaggle/input/**/rftm/__init__.py", recursive=True))
if attached:
    REPO = os.path.dirname(os.path.dirname(attached[0]))
    print("using the attached copy of the repo:", REPO)
else:
    REPO = "/kaggle/working/restoring-for-the-machine"
    if os.path.exists(REPO):                       # a clone from an earlier run in this session: update it
        subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
    print("cloned", REPO_URL, "@", subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                                                    capture_output=True, text=True).stdout.strip())
sys.path.insert(0, REPO)

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

from rftm.degradations import DEGRADATIONS, degrade
from rftm import restorers as RS, coco as C, taskloss as TL, detectors as D, analysis8 as B
from rftm.kaggle import resolve_checkpoint

QUICK = False                  # True: tiny sweep (lambda 0 and 0.3, 60 steps, 40 images) to check the pipeline
SEED = 0
LAMS = [0, 0.3] if QUICK else [0, 0.03, 0.1, 0.3, 1, 3]
STEPS = 60 if QUICK else 2000
DROP_AT = 45 if QUICK else 1500
N_TEST = 40 if QUICK else 300
N_VAL = 40 if QUICK else 500
SIGMAS = [0, 50] if QUICK else [0, 15, 25, 50]
OUT_DIR = "/kaggle/working/outputs/08" + ("_quick" if QUICK else ""); os.makedirs(OUT_DIR, exist_ok=True)
CK_DIR = f"{OUT_DIR}/checkpoints"; os.makedirs(CK_DIR, exist_ok=True)
COCO_DIR = "/kaggle/working/coco"
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "turn on Settings -> Accelerator -> GPU T4"
PICK = json.load(open(f"{REPO}/results/07/layers_for_partB.json"))     # Part A's pre-registered choice
LOSS_LAYERS, LOSS_METRIC = [int(l) for l in PICK["layers"]], PICK["metric"]
NECK = [15, 18, 21]
print("feature loss:", LOSS_METRIC, "at layers", LOSS_LAYERS, f"(held-out rho {PICK['rho_odd']:.3f}) | QUICK:", QUICK)

## 1 · Splits

- **test:** notebook 05's 300 images (checked against the frozen id list), with the same noise seeds;
- **train / val:** the other 4,700 val2017 images, shuffled with seed 0: 4,200 for training, 500 for choosing λ.

YOLOv8n was trained on COCO *train*, so none of these images were seen by any detector.

In [ ]:
C.download_coco(COCO_DIR)
coco_gt = C.load_coco_gt(COCO_DIR)
test_all = C.subset_ids(coco_gt, 300, SEED)
POS = {iid: k for k, iid in enumerate(test_all)}
test_ids = test_all[:N_TEST]
rest = sorted(set(coco_gt.getImgIds()) - set(test_all))
random.Random(SEED).shuffle(rest)
train_ids, val_all = rest[:4200], rest[4200:]
val_all = [i for i in val_all if len(coco_gt.getAnnIds(imgIds=i, iscrowd=False)) > 0]   # mAP needs objects
val_ids = sorted(val_all[:N_VAL])
assert not (set(train_ids) & set(test_all)) and not (set(val_ids) & set(test_all)) and not (set(train_ids) & set(val_ids))
json.dump({"train": train_ids, "val": val_ids, "test": test_ids}, open(f"{OUT_DIR}/splits.json", "w"))
path = lambda i: f"{COCO_DIR}/val2017/{coco_gt.loadImgs(i)[0]['file_name']}"
print(f"train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)} images")

## 2 · Detectors and the starting DnCNN

YOLOv8n is scored exactly as in notebooks 05/07 (and must give 0.387 on the clean test images). YOLOv8s and Faster
R-CNN R50-FPN v2 are only ever used for testing.

In [ ]:
net0, ck = RS.load_dncnn(resolve_checkpoint(), device)
print(f"starting DnCNN: {ck['epoch'] + 1} epochs of blind Gaussian training (notebook 04)")
DETECTORS = {"yolov8n": D.UltralyticsDetector("yolov8n.pt", coco_gt, device),
             "yolov8s": D.UltralyticsDetector("yolov8s.pt", coco_gt, device),
             "frcnn_r50_v2": D.FasterRCNN(coco_gt, device)}
test_imgs = {i: C.load_coco_rgb(coco_gt, COCO_DIR, i) for i in test_ids}
for name, det in DETECTORS.items():
    t0 = time.time(); res = []
    for i in test_ids:
        res += det(test_imgs[i], i)
    print(f"{name:13s} clean test mAP {C.coco_eval(coco_gt, res, test_ids)[0]:.3f}  ({time.time() - t0:.0f} s)")
    if name == "yolov8n" and not QUICK:
        assert abs(C.coco_eval(coco_gt, res, test_ids)[0] - 0.387) < 0.003, "not the same setup as notebooks 05/07"

## 3 · The differentiable YOLO forward

`TL.YoloFeatures` runs YOLOv8n's layers itself (no `predict()`, no NMS) so that gradients can flow through it. It uses
its own copy of YOLOv8n, frozen. Three checks:
1. its layer-21 output equals what YOLOv8n's own forward produces at that layer (via a forward hook);
2. gradients reach the input image but not YOLO's weights;
3. the feature loss is 0 for identical images and grows with noise.

In [ ]:
yolo_for_loss = YOLO("yolov8n.pt").model.float().to(device)
yf = TL.YoloFeatures(yolo_for_loss, sorted(set(LOSS_LAYERS) | set(NECK)))

ref = YOLO("yolov8n.pt").model.float().to(device).eval()
caught = {}
h = ref.model[21].register_forward_hook(lambda m, a, o: caught.__setitem__(21, o.detach()))
x = torch.from_numpy(np.ascontiguousarray(test_imgs[test_ids[0]][:320, :320].transpose(2, 0, 1)))[None].to(device)
with torch.no_grad():
    ref(x); mine = yf(x)[21]
h.remove()
err = (mine - caught[21]).abs().max().item()
print(f"1. layer 21: max |ours - ultralytics| = {err:.2e}"); assert err < 1e-4

fl = TL.FeatureLoss(TL.YoloFeatures(yolo_for_loss, LOSS_LAYERS), LOSS_METRIC)
xg = (x + 25 / 255 * torch.randn_like(x)).clamp(0, 1).requires_grad_(True)   # a stand-in "restored" image
fl(xg, x).backward()
print(f"2. input gradient norm {xg.grad.norm().item():.3e}; YOLO weights with gradients: "
      f"{sum(p.grad is not None for p in yolo_for_loss.parameters())}")
assert xg.grad.norm().item() > 0 and all(p.grad is None for p in yolo_for_loss.parameters())

with torch.no_grad():
    d = [fl((x + s / 255 * torch.randn_like(x)).clamp(0, 1), x).item() for s in (0, 10, 50)]
print("3. feature loss at sigma 0 / 10 / 50:", [round(v, 4) for v in d]); assert d[0] < 1e-6 < d[1] < d[2]

## 4 · Training data and loss scales

MSE₀ and D₀ are the two terms' mean values for the *starting* DnCNN on 20 fixed training batches, so that at λ = 1
the feature term starts with the same weight as the pixel term. The same constants are used for every λ.

In [ ]:
train_ds = TL.CocoCrops([path(i) for i in train_ids], crop=320)
feat_loss = TL.FeatureLoss(TL.YoloFeatures(yolo_for_loss, LOSS_LAYERS), LOSS_METRIC)
neck_loss = TL.FeatureLoss(TL.YoloFeatures(yolo_for_loss, NECK), "cos")
SCALES = f"{OUT_DIR}/loss_scales.json"
if os.path.exists(SCALES):
    scales = json.load(open(SCALES))
else:
    mse0, d0 = TL.loss_scales(net0, feat_loss, TL.make_loader(train_ds, 8, seed=999), device)
    _, d0_neck = TL.loss_scales(net0, neck_loss, TL.make_loader(train_ds, 8, seed=999), device)
    scales = dict(mse0=mse0, d0=d0, d0_neck=d0_neck)
    json.dump(scales, open(SCALES, "w"))
print({k: round(v, 6) for k, v in scales.items()}, "| starting train PSNR", round(10 * np.log10(1 / scales["mse0"]), 2), "dB")

## 5 · The λ sweep

Every run starts from `dncnn_last.pt`, uses the same seed (so the same crops and noise in the same order) and the same
step budget. λ = 0 is the control: the same fine-tuning on COCO crops with MSE only. Checkpoints are saved, so a
restarted session skips finished runs.

In [ ]:
def ck_path(name): return f"{CK_DIR}/{name}.pt"
models = {"orig": net0}
for lam in LAMS:
    name = f"lam_{lam:g}"
    if os.path.exists(ck_path(name)):
        net = RS.DnCNN(**ck["cfg"]).to(device); net.load_state_dict(torch.load(ck_path(name), map_location=device)); net.eval()
        print(f"{name}: loaded"); models[name] = net; continue
    t0 = time.time()
    net, hist = TL.train_dncnn(net0, feat_loss, TL.make_loader(train_ds, 8, seed=SEED), device, lam,
                               scales["mse0"], scales["d0"], steps=STEPS, drop_at=DROP_AT, seed=SEED,
                               log_every=max(10, STEPS // 10))
    torch.save(net.state_dict(), ck_path(name)); pd.DataFrame(hist).to_csv(f"{CK_DIR}/{name}_log.csv", index=False)
    models[name] = net
    print(f"{name}: trained in {(time.time() - t0) / 60:.1f} min")

## 6 · Choosing λ\* on the validation split

The rule from PLAN.md §2.1: the λ > 0 with the highest YOLOv8n mAP on the 500 validation images at σ = 50, among
those whose validation PSNR at σ = 50 is at most 1 dB below the control's. The test images are not used for this.

In [ ]:
gauss_idx = list(DEGRADATIONS).index("gaussian")
def noisy_version(img, sigma, key):
    # Notebook 05's noise for test images (key = position in the 300-image subset); val images use key 10000 + k.
    if sigma == 0:
        return img
    rng = np.random.default_rng([SEED, key, gauss_idx, DEGRADATIONS["gaussian"][1].index(sigma)])
    return degrade(img, "gaussian", sigma, rng)

RUNS_CSV, PI_CSV = f"{OUT_DIR}/runs.csv", f"{OUT_DIR}/per_image.csv"
runs_rows = pd.read_csv(RUNS_CSV).to_dict("records") if os.path.exists(RUNS_CSV) else []
pi_rows = pd.read_csv(PI_CSV).to_dict("records") if os.path.exists(PI_CSV) else []
done = {(r["split"], r["model"], r["detector"], r["sigma"]) for r in runs_rows}

def evaluate(split, mname, sigma, ids, imgs, keys, detectors):
    todo = [d for d in detectors if (split, mname, d, sigma) not in done]
    if not todo:
        return
    t0 = time.time()
    restore = None if mname == "noisy" else RS.make_dncnn_restore(models[mname], device)
    outs, psnrs = {}, {}
    for i in ids:
        y = noisy_version(imgs[i], sigma, keys[i])
        out = y if restore is None else restore(y)
        outs[i] = out
        psnrs[i] = 10 * np.log10(1 / max(float(np.mean((out - imgs[i]) ** 2)), 1e-12))
    for dname in todo:
        res = []
        for i in ids:
            res += DETECTORS[dname](outs[i], i)
        st = C.coco_eval(coco_gt, res, ids)
        if split == "test":
            for i, ap in C.per_image_ap(coco_gt, res, ids).items():
                pi_rows.append(dict(model=mname, detector=dname, sigma=sigma, image_id=i, ap=ap, psnr=psnrs[i]))
            pd.DataFrame(pi_rows).to_csv(PI_CSV, index=False)
        runs_rows.append(dict(split=split, model=mname, detector=dname, sigma=sigma, map=float(st[0]), map50=float(st[1]),
                              psnr=float(np.mean([p for p in psnrs.values() if np.isfinite(p)] or [np.inf]))))
        done.add((split, mname, dname, sigma)); pd.DataFrame(runs_rows).to_csv(RUNS_CSV, index=False)
        print(f"{split:4s} {mname:10s} sigma {sigma:2d} {dname:13s} mAP {st[0]:.3f}  PSNR {runs_rows[-1]['psnr']:.2f}"
              f"  ({time.time() - t0:.0f} s)")

val_imgs = {i: C.load_coco_rgb(coco_gt, COCO_DIR, i) for i in val_ids}
val_keys = {i: 10000 + k for k, i in enumerate(val_ids)}
for name in [f"lam_{l:g}" for l in LAMS]:
    evaluate("val", name, 50, val_ids, val_imgs, val_keys, ["yolov8n"])
runs = pd.DataFrame(runs_rows)
star, sel = B.choose_lambda_star(runs)
print(sel.round(4).to_string()); print("lambda* =", star)
json.dump({"lambda_star": star, "table": sel.reset_index().to_dict("records")}, open(f"{OUT_DIR}/lambda_star.json", "w"), indent=1)
del val_imgs

## 7 · Comparison: the neck-mean loss at λ\*

Part A's three best options were within 0.01 of each other, so one extra model uses the mean cosine distance over the
three neck outputs (15, 18, 21) at the same λ\*, with its own D₀. Exploratory: no hypothesis rides on it.

In [ ]:
if star is not None:
    lam_star = B.lam_of(star); name = f"neck_{lam_star:g}"
    if os.path.exists(ck_path(name)):
        net = RS.DnCNN(**ck["cfg"]).to(device); net.load_state_dict(torch.load(ck_path(name), map_location=device)); net.eval()
    else:
        net, hist = TL.train_dncnn(net0, neck_loss, TL.make_loader(train_ds, 8, seed=SEED), device, lam_star,
                                   scales["mse0"], scales["d0_neck"], steps=STEPS, drop_at=DROP_AT, seed=SEED,
                                   log_every=max(10, STEPS // 10))
        torch.save(net.state_dict(), ck_path(name)); pd.DataFrame(hist).to_csv(f"{CK_DIR}/{name}_log.csv", index=False)
    models[name] = net
print("models:", list(models))

## 8 · Test

All models × σ ∈ {0, 15, 25, 50} × three detectors on the 300 test images, with the noise of notebooks 05/07.
"noisy" = no restoration. Results are saved after every (model, σ, detector), so a restart resumes.

In [ ]:
test_keys = {i: POS[i] for i in test_ids}
order = ["noisy"] + list(models)
t_all = time.time()
for sigma in SIGMAS:
    for mname in order:
        if mname == "noisy" and sigma == 0:
            continue                                  # clean images: covered by the clean baselines above
        evaluate("test", mname, sigma, test_ids, test_imgs, test_keys, list(DETECTORS))
print(f"test done in {(time.time() - t_all) / 60:.1f} min")

## 9 · Results

1. mAP for each detector, σ and model; PSNR against mAP across λ (the trade-off curve);
2. paired bootstrap of every model against the λ = 0 control;
3. exploratory: share of the restoration error's energy at high spatial frequencies (adversarial-looking patterns
   would show up here);
4. the scorecard for H7–H10.

In [ ]:
runs = pd.read_csv(RUNS_CSV); pi = pd.read_csv(PI_CSV)
tt = B.test_table(runs)
cols = [c for c in ["noisy", "orig"] + [f"lam_{l:g}" for l in LAMS] + [c for c in tt.columns if c.startswith("neck_")] if c in tt.columns]
print("test mAP@[.5:.95]:"); print(tt[cols].round(3).to_string())
pt = runs[runs.split == "test"].pivot_table(index="sigma", columns="model", values="psnr")
print("\ntest PSNR (dB):"); print(pt[[c for c in cols if c in pt.columns]].round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
lam_cols = [f"lam_{l:g}" for l in LAMS]
for ax, det in zip(axes, DETECTORS):
    for sigma, c in zip([s for s in SIGMAS if s > 0], ["#9ecae1", "#4292c6", "#08519c"]):
        sub = runs[(runs.split == "test") & (runs.detector == det) & (runs.sigma == sigma)].set_index("model")
        xs, ys = sub.loc[lam_cols, "psnr"], sub.loc[lam_cols, "map"]
        ax.plot(xs, ys, "o-", color=c, label=f"σ = {sigma}")
        for m, x_, y_ in zip(lam_cols, xs, ys):
            ax.annotate(m[4:], (x_, y_), fontsize=7, xytext=(3, 3), textcoords="offset points")
        if "orig" in sub.index:
            ax.plot(sub.loc["orig", "psnr"], sub.loc["orig", "map"], "k*", ms=9)
    ax.set_title(det); ax.set_xlabel("PSNR (dB)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("test mAP@[.5:.95]"); axes[0].legend(fontsize=8)
fig.suptitle("PSNR vs mAP across λ (labels = λ; ★ = the starting DnCNN). Up-left = better detection for less PSNR.")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_psnr_vs_map.png", dpi=150); plt.show()

cmp_ = B.comparisons(pi, [m for m in cols if m != "noisy"], list(DETECTORS), [s for s in SIGMAS])
cmp_.round(4).to_csv(f"{OUT_DIR}/comparisons.csv", index=False)
print("\nimage-averaged AP vs the lambda = 0 control (95% bootstrap CI):")
print(cmp_[cmp_.sigma == max(SIGMAS)].round(4).to_string(index=False))

In [ ]:
# Exploratory: high-frequency share of the restoration error at sigma = 50, on 20 test images
hf = []
for mname in [m for m in ["orig", "lam_0", star, f"lam_{max(LAMS):g}"] if m in models]:
    restore = RS.make_dncnn_restore(models[mname], device)
    for i in test_ids[:20]:
        out = restore(noisy_version(test_imgs[i], 50, POS[i]))
        hf.append(dict(model=mname, image_id=i, hf_share=TL.high_freq_share(out - test_imgs[i])))
hf = pd.DataFrame(hf); hf.to_csv(f"{OUT_DIR}/high_freq.csv", index=False)
print("share of error energy above half-Nyquist, sigma 50 (mean over 20 images):")
print(hf.groupby("model").hf_share.mean().round(4).to_string())

i0 = test_ids[0]
show = [m for m in ["orig", "lam_0", star, f"lam_{max(LAMS):g}"] if m in models]
fig, axes = plt.subplots(1, len(show) + 2, figsize=(3.2 * (len(show) + 2), 3.4))
y = noisy_version(test_imgs[i0], 50, POS[i0])
H, W = y.shape[:2]; c = min(200, H, W); y0, x0 = (H - c) // 2, (W - c) // 2
crop = lambda im: im[y0:y0 + c, x0:x0 + c]                      # centre crop, at most 200 x 200
axes[0].imshow(crop(test_imgs[i0])); axes[0].set_title("clean")
axes[1].imshow(crop(y)); axes[1].set_title("σ = 50")
for ax, m in zip(axes[2:], show):
    ax.imshow(crop(RS.make_dncnn_restore(models[m], device)(y))); ax.set_title(m)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_crops.png", dpi=150); plt.show()

In [ ]:
def to_py(x):
    if isinstance(x, dict):  return {str(k): to_py(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):  return [to_py(v) for v in x]
    if isinstance(x, (np.floating, float)):  return None if not np.isfinite(x) else round(float(x), 4)
    if isinstance(x, (np.integer,)):  return int(x)
    if isinstance(x, (np.bool_,)):  return bool(x)
    return x

sc = B.scorecard8(runs, pi, star)
for k, (v, ev) in sc.items():
    print(f"{k:4s} {str(v):5s}  {json.dumps(to_py(ev))[:260]}")
json.dump(to_py({"QUICK": QUICK, "lambda_star": star, "loss": {"metric": LOSS_METRIC, "layers": LOSS_LAYERS},
                 "scales": scales, "verdicts": {k: v for k, (v, _) in sc.items()},
                 "evidence": {k: e for k, (_, e) in sc.items()}}), open(f"{OUT_DIR}/scorecard_08.json", "w"), indent=1)

## 10 · Clean-up

Deletes the COCO download, the cloned repo and the detector weights from `/kaggle/working`, so the notebook's output
holds only `outputs/08` (small enough to download in one go, also with `kaggle kernels output`).

In [ ]:
import shutil
for p in [COCO_DIR, "/kaggle/working/restoring-for-the-machine", "/kaggle/working/dncnn_last.pt"] + glob.glob("/kaggle/working/*.pt"):
    if os.path.isdir(p): shutil.rmtree(p, ignore_errors=True)
    elif os.path.exists(p): os.remove(p)
print("left in /kaggle/working:", os.listdir("/kaggle/working"))
print("outputs:", sorted(os.listdir(OUT_DIR)))

## Findings

*Written after the run, from the scorecard and the figures above.*